# SonicForge — Development Sandbox

Use this notebook to quickly test libraries, API calls, and ideas **without touching the main codebase**.

Run it with the project's virtual environment kernel:
```
source ../.venv/bin/activate
jupyter notebook
```

## 1. Read FLAC Metadata with mutagen

In [ ]:
from mutagen.flac import FLAC

# Replace with any local FLAC file path
path = "/path/to/your/file.flac"

audio = FLAC(path)
for key, val in audio.items():
    print(f"{key:20} = {val}")

print(f"\nPictures: {len(audio.pictures)}")

## 2. Fetch Artwork from iTunes Search API

In [ ]:
import requests
import urllib.parse
from IPython.display import Image, display

artist = "Adele"
album  = "19"

term  = urllib.parse.quote_plus(f"{artist} {album}")
url   = f"https://itunes.apple.com/search?term={term}&entity=album&limit=5"
data  = requests.get(url, timeout=5).json()

for r in data.get('results', []):
    thumb = r.get('artworkUrl100', '')
    print(r.get('collectionName'), '->', thumb)
    if thumb:
        display(Image(url=thumb))

## 3. Fetch Artwork from Deezer API

In [ ]:
import requests
from IPython.display import Image, display

artist = "Adele"
album  = "19"

url  = f'https://api.deezer.com/search/album?q=artist:"{artist}" album:"{album}"&limit=5'
data = requests.get(url, timeout=5).json()

for r in data.get('data', []):
    cover = r.get('cover_medium', '')
    print(r.get('title'), '->', cover)
    if cover:
        display(Image(url=cover))

## 4. Image Processing — Crop & Resize to 500×500 JPEG

In [ ]:
import io, requests
from PIL import Image as PILImage
from IPython.display import Image, display

img_url = "https://is1-ssl.mzstatic.com/image/thumb/Music/v4/3b/4c/f1/3b4cf100x100bb.jpg"

raw = requests.get(img_url, timeout=5).content
img = PILImage.open(io.BytesIO(raw))

# Centre-crop to square
w, h = img.size
m = min(w, h)
img = img.crop(((w - m) // 2, (h - m) // 2, (w + m) // 2, (h + m) // 2))
img = img.resize((500, 500), PILImage.Resampling.LANCZOS)
if img.mode != 'RGB':
    img = img.convert('RGB')

buf = io.BytesIO()
img.save(buf, format='JPEG')
display(Image(data=buf.getvalue()))
print(f"Output size: {len(buf.getvalue())} bytes")

## 5. Write Metadata Back to a FLAC File

In [ ]:
from mutagen.flac import FLAC

# ⚠️  This WILL modify the file on disk!
path = "/path/to/your/file.flac"

audio = FLAC(path)
audio['artist'] = 'New Artist'
audio['album']  = 'New Album'
audio.save()

print("Tags saved.")

## 6. Query Release Search on MusicBrainz API

In [ ]:
import requests
import urllib.parse

artist = "Adele"
album  = "21"

headers = {
    "User-Agent": "SonicForgeSandbox/1.0.0 ( contact@example.com )"
}

parts = []
if artist:
    parts.append(f'artist:"{artist}"')
if album:
    parts.append(f'release:"{album}"')
query = " AND ".join(parts)
encoded_query = urllib.parse.quote_plus(query)

url = f"https://musicbrainz.org/ws/2/release/?query={encoded_query}&fmt=json&limit=5"
resp = requests.get(url, headers=headers, timeout=5)
data = resp.json()

for r in data.get('releases', []):
    print(f"Title:  {r.get('title')}")
    print(f"ID:     {r.get('id')}")
    print(f"Tracks: {r.get('track-count')}")
    print(f"Date:   {r.get('date', 'Unknown')}")
    label_info = r.get('label-info-list', [])
    label_name = 'Unknown'
    if label_info and label_info[0].get('label'):
        label_name = label_info[0]['label'].get('name', 'Unknown')
    print(f"Label:  {label_name}")
    print(f"Thumb:  https://coverartarchive.org/release/{r.get('id')}/front-250\n")

## 7. Retrieve Tracklist Details from MusicBrainz API

In [ ]:
import requests

# Release ID for Adele's 21
release_id = "68019ab7-b892-4874-9844-32a265691060"

headers = {
    "User-Agent": "SonicForgeSandbox/1.0.0 ( contact@example.com )"
}

url = f"https://musicbrainz.org/ws/2/release/{release_id}?inc=recordings+artists&fmt=json"
resp = requests.get(url, headers=headers, timeout=5)
data = resp.json()

for media in data.get('media', []):
    print(f"--- Format: {media.get('format', 'CD')} ---")
    for track in media.get('tracks', []):
        title = track.get('title')
        num = track.get('number')
        length_ms = track.get('length') or track.get('recording', {}).get('length', 0)
        duration_str = ""
        if length_ms:
            total_sec = length_ms // 1000
            minutes = total_sec // 60
            seconds = total_sec % 60
            duration_str = f"{minutes:02d}:{seconds:02d}"
            
        print(f"{num:2}. {title} {duration_str}")

## 8. Parse CUE Sheet (.cue) in Python

In [ ]:
def parse_time(time_str):
    parts = time_str.split(':')
    mins, secs, frames = int(parts[0]), int(parts[1]), int(parts[2])
    return mins * 60.0 + secs + frames / 75.0

cue_data = """PERFORMER "Adele"
TITLE "21"
FILE "Adele - 21.flac" WAVE
  TRACK 01 AUDIO
    TITLE "Intro"
    INDEX 01 00:00:00
  TRACK 02 AUDIO
    TITLE "Rolling in the Deep"
    INDEX 01 02:30:15"""

tracks = []
current = None
for line in cue_data.splitlines():
    line = line.strip()
    if line.startswith("TRACK"):
        current = {'num': line.split()[1], 'title': '', 'start': 0.0}
        tracks.append(current)
    elif line.startswith("TITLE") and current:
        current['title'] = line.split('"')[1]
    elif line.startswith("INDEX 01") and current:
        current['start'] = parse_time(line.split()[2])

for t in tracks:
    print(f"Track {t['num']}: {t['title']:25} Starts at {t['start']}s")

## 9. Slicing continuous Audio via FFmpeg

In [ ]:
import subprocess

# ⚠️ This requires a continuous FLAC file path to test
large_flac = "/path/to/your/album_large_file.flac"
output_flac = "/path/to/your/track_01_slice.flac"

start_seconds = 0.0
duration_seconds = 150.2

cmd = [
    'ffmpeg', '-y',
    '-i', large_flac,
    '-ss', str(start_seconds),
    '-t', str(duration_seconds),
    '-c:a', 'flac',
    output_flac
]

print(f"Running command: {' '.join(cmd)}")
# subprocess.run(cmd, check=True)

## 10. Transcoding Audio to MP3 / OGG / WAV via FFmpeg

In [ ]:
import subprocess

# Example: Transcode FLAC to MP3 (320 kbps)
input_audio = "/path/to/your/input.flac"
output_mp3  = "/path/to/your/output.mp3"

cmd_mp3 = [
    'ffmpeg', '-y',
    '-i', input_audio,
    '-c:a', 'libmp3lame',
    '-b:a', '320k',
    output_mp3
]

print(f"MP3 Command: {' '.join(cmd_mp3)}")
# subprocess.run(cmd_mp3, check=True)

# Example: Transcode FLAC to OGG Vorbis (quality level q6)
output_ogg = "/path/to/your/output.ogg"
cmd_ogg = [
    'ffmpeg', '-y',
    '-i', input_audio,
    '-c:a', 'libvorbis',
    '-q:a', '6',
    output_ogg
]

print(f"OGG Command: {' '.join(cmd_ogg)}")
# subprocess.run(cmd_ogg, check=True)

# Example: Transcode FLAC to WAV (PCM 16-bit)
output_wav = "/path/to/your/output.wav"
cmd_wav = [
    'ffmpeg', '-y',
    '-i', input_audio,
    '-c:a', 'pcm_s16le',
    output_wav
]

print(f"WAV Command: {' '.join(cmd_wav)}")
# subprocess.run(cmd_wav, check=True)